# Exp3 机器翻译实验 PyTorch Notebook

这个 notebook 按手册要求实现 `Seq2Seq + GRU + Attention` 英译中实验，并支持一次性跑完 `preprocess -> train -> eval -> predict` 四个流程。

In [ ]:
import json
import random
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

RUN_CONFIG = {
    "command": "all",  # all / preprocess / train / eval / predict
    "data_file": "./ReferenceCode/src/cmn_zhsim.txt",
    "preprocess_dir": "./preprocess_pytorch",
    "checkpoint_dir": "./checkpoints_pytorch",
    "checkpoint_name": "seq2seq_attention_gru.pt",
    "log_dir": "./logs",
    "run_name": None,
    "device": "auto",  # auto / cuda / mps / cpu
    "seed": 42,
    "num_samples": 20000,
    "max_seq_length": 10,
    "hidden_size": 512,
    "batch_size": 16,
    "eval_batch_size": 1,
    "learning_rate": 1e-3,
    "num_epochs": 15,
    "train_split": 0.8,
    "eval_log_samples": 5,
    "predict_sentence": "i am a student .",
}

PAD_ID = 0
EOS_ID = 1
SOS_ID = 2
PAD_TOKEN = "<pad>"
EOS_TOKEN = "<eos>"
SOS_TOKEN = "<sos>"
LOG_FILE = None


def resolve_device(device_name: str) -> torch.device:
    if device_name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(device_name)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_args(config):
    normalized = dict(config)
    for key in {"data_file", "preprocess_dir", "checkpoint_dir", "log_dir"}:
        normalized[key] = Path(normalized[key])
    return type("Config", (), normalized)()


def config_to_dict(args):
    data = {}
    for key, value in vars(args.__class__).items():
        if key.startswith("__") or callable(value):
            continue
        data[key] = str(value) if isinstance(value, Path) else value
    return data


def setup_log_file(args):
    global LOG_FILE
    args.log_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = args.run_name or f"translation_{timestamp}"
    LOG_FILE = args.log_dir / f"{run_name}.log"
    LOG_FILE.write_text("", encoding="utf-8")
    return LOG_FILE


def log(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {message}"
    print(line)
    if LOG_FILE is not None:
        with LOG_FILE.open("a", encoding="utf-8") as f:
            f.write(line + "\n")


args = build_args(RUN_CONFIG)
device = resolve_device(args.device)
print("current device:", device)

In [ ]:
def unicode_to_ascii(text: str) -> str:
    return "".join(
        char for char in unicodedata.normalize("NFD", text) if unicodedata.category(char) != "Mn"
    )


def normalize_english(text: str) -> str:
    text = unicode_to_ascii(text.lower().strip())
    text = re.sub(r"([.!?])", r" \1", text)
    text = re.sub(r"[^a-zA-Z.!?]+", r" ", text)
    return re.sub(r"\s+", " ", text).strip()


def pad_or_truncate(ids, max_seq_len):
    max_total_len = max_seq_len + 1
    if len(ids) <= max_total_len:
        return ids + [PAD_ID] * (max_total_len - len(ids))
    return ids[:max_seq_len] + [PAD_ID]


def save_vocab(vocab_path: Path, tokens) -> None:
    vocab_path.write_text("\n".join(tokens), encoding="utf-8")


def load_vocab(vocab_path: Path):
    return vocab_path.read_text(encoding="utf-8").splitlines()


def preprocess_dataset(args):
    data_file = args.data_file
    output_dir = args.preprocess_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    lines = data_file.read_text(encoding="utf-8").splitlines()
    pairs = []
    for line in lines:
        if "\t" not in line:
            continue
        src, tgt = line.split("\t", 1)
        src = normalize_english(src)
        tgt = tgt.strip()
        if src and tgt:
            pairs.append((src, tgt))

    pairs = pairs[: args.num_samples]
    random.Random(args.seed).shuffle(pairs)
    english_sentences = [pair[0] for pair in pairs]
    chinese_sentences = [pair[1] for pair in pairs]

    en_vocab = sorted({token for sentence in english_sentences for token in sentence.split() if token})
    ch_vocab = sorted(set("".join(chinese_sentences)))
    id2en = [PAD_TOKEN, EOS_TOKEN, SOS_TOKEN] + en_vocab
    id2ch = [PAD_TOKEN, EOS_TOKEN, SOS_TOKEN] + ch_vocab
    en2id = {token: idx for idx, token in enumerate(id2en)}
    ch2id = {token: idx for idx, token in enumerate(id2ch)}

    encoder_data = []
    for sentence in english_sentences:
        ids = [SOS_ID] + [en2id[token] for token in sentence.split()] + [EOS_ID]
        encoder_data.append(pad_or_truncate(ids, args.max_seq_length))

    decoder_data = []
    for sentence in chinese_sentences:
        ids = [SOS_ID] + [ch2id[token] for token in sentence] + [EOS_ID]
        decoder_data.append(pad_or_truncate(ids, args.max_seq_length))

    encoder_data = np.asarray(encoder_data, dtype=np.int64)
    decoder_data = np.asarray(decoder_data, dtype=np.int64)

    total = len(encoder_data)
    train_size = max(1, int(total * args.train_split))
    train_size = min(train_size, total - 1) if total > 1 else total
    train_encoder = encoder_data[:train_size]
    train_decoder = decoder_data[:train_size]
    eval_encoder = encoder_data[train_size:] if total > train_size else encoder_data[: min(20, total)]
    eval_decoder = decoder_data[train_size:] if total > train_size else decoder_data[: min(20, total)]

    np.savez(output_dir / "gru_train.npz", encoder_data=train_encoder, decoder_data=train_decoder)
    np.savez(output_dir / "gru_eval.npz", encoder_data=eval_encoder, decoder_data=eval_decoder)
    save_vocab(output_dir / "en_vocab.txt", id2en)
    save_vocab(output_dir / "ch_vocab.txt", id2ch)

    log(f"preprocess done: {output_dir}")
    log(f"total pairs: {total}")
    log(f"train pairs: {len(train_encoder)}")
    log(f"eval pairs: {len(eval_encoder)}")
    log(f"en vocab size: {len(id2en)}")
    log(f"ch vocab size: {len(id2ch)}")


def ensure_preprocessed(args):
    required = [
        args.preprocess_dir / "gru_train.npz",
        args.preprocess_dir / "gru_eval.npz",
        args.preprocess_dir / "en_vocab.txt",
        args.preprocess_dir / "ch_vocab.txt",
    ]
    if all(path.exists() for path in required):
        log(f"reuse preprocess data: {args.preprocess_dir}")
        return
    preprocess_dataset(args)

In [ ]:
def create_dataloader(args, is_training: bool):
    data_path = args.preprocess_dir / ("gru_train.npz" if is_training else "gru_eval.npz")
    with np.load(data_path) as data:
        encoder_data = data["encoder_data"]
        decoder_data = data["decoder_data"]

    if is_training:
        encoder_input = encoder_data[:, :-1]
        decoder_input = decoder_data[:, :-1]
        decoder_target = decoder_data[:, 1:]
        dataset = TensorDataset(
            torch.from_numpy(encoder_input),
            torch.from_numpy(decoder_input),
            torch.from_numpy(decoder_target),
        )

        def collate_fn(batch):
            return {
                "encoder_data": torch.stack([item[0] for item in batch]),
                "decoder_data": torch.stack([item[1] for item in batch]),
                "target_data": torch.stack([item[2] for item in batch]),
            }

        return DataLoader(
            dataset,
            batch_size=args.batch_size,
            shuffle=True,
            drop_last=True,
            collate_fn=collate_fn,
        )

    encoder_input = encoder_data[:, :-1]
    decoder_input = decoder_data[:, :-1]
    decoder_target = decoder_data[:, 1:]
    dataset = TensorDataset(
        torch.from_numpy(encoder_input),
        torch.from_numpy(decoder_input),
        torch.from_numpy(decoder_target),
    )

    def collate_fn(batch):
        return {
            "encoder_data": torch.stack([item[0] for item in batch]),
            "decoder_data": torch.stack([item[1] for item in batch]),
            "target_data": torch.stack([item[2] for item in batch]),
        }

    return DataLoader(
        dataset,
        batch_size=args.eval_batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=collate_fn,
    )

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.encoder_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.decoder_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.score_proj = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, src_mask):
        query = decoder_hidden[-1]
        scores = self.score_proj(
            torch.tanh(self.encoder_proj(encoder_outputs) + self.decoder_proj(query).unsqueeze(1))
        ).squeeze(-1)
        scores = scores.masked_fill(~src_mask, -1e9)
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        return context, attn_weights


class Seq2SeqAttentionGRU(nn.Module):
    def __init__(self, en_vocab_size, ch_vocab_size, hidden_size, max_seq_length):
        super().__init__()
        self.hidden_size = hidden_size
        self.max_seq_length = max_seq_length
        self.encoder_embedding = nn.Embedding(en_vocab_size, hidden_size)
        self.decoder_embedding = nn.Embedding(ch_vocab_size, hidden_size)
        self.encoder_gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.attention = AdditiveAttention(hidden_size)
        self.decoder_gru = nn.GRU(hidden_size * 2, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size * 2, ch_vocab_size)

    def encode(self, src):
        embedded = self.encoder_embedding(src)
        encoder_outputs, hidden = self.encoder_gru(embedded)
        src_mask = src.ne(PAD_ID)
        return encoder_outputs, hidden, src_mask

    def decode_step(self, decoder_input, hidden, encoder_outputs, src_mask):
        embedded = self.decoder_embedding(decoder_input)
        context, attn_weights = self.attention(hidden, encoder_outputs, src_mask)
        decoder_rnn_input = torch.cat([embedded, context], dim=-1)
        decoder_output, hidden = self.decoder_gru(decoder_rnn_input, hidden)
        logits = self.output_layer(torch.cat([decoder_output, context], dim=-1))
        return logits, hidden, attn_weights

    def forward(self, src, tgt):
        encoder_outputs, hidden, src_mask = self.encode(src)
        logits_steps = []
        for step in range(tgt.size(1)):
            decoder_input = tgt[:, step : step + 1]
            logits, hidden, _ = self.decode_step(decoder_input, hidden, encoder_outputs, src_mask)
            logits_steps.append(logits)
        return torch.cat(logits_steps, dim=1)

    def greedy_decode(self, src, max_len):
        encoder_outputs, hidden, src_mask = self.encode(src)
        decoder_input = torch.full((src.size(0), 1), SOS_ID, dtype=torch.long, device=src.device)
        outputs = []
        finished = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
        for _ in range(max_len):
            logits, hidden, _ = self.decode_step(decoder_input, hidden, encoder_outputs, src_mask)
            next_token = logits.argmax(dim=-1)
            outputs.append(next_token)
            finished |= next_token.squeeze(1).eq(EOS_ID)
            if torch.all(finished):
                break
            decoder_input = next_token
        return torch.cat(outputs, dim=1)


def load_meta(args):
    en_vocab = load_vocab(args.preprocess_dir / "en_vocab.txt")
    ch_vocab = load_vocab(args.preprocess_dir / "ch_vocab.txt")
    return en_vocab, ch_vocab


def build_model(args):
    en_vocab, ch_vocab = load_meta(args)
    model = Seq2SeqAttentionGRU(
        en_vocab_size=len(en_vocab),
        ch_vocab_size=len(ch_vocab),
        hidden_size=args.hidden_size,
        max_seq_length=args.max_seq_length,
    )
    return model, en_vocab, ch_vocab


def save_checkpoint(model, args, en_vocab, ch_vocab):
    args.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = args.checkpoint_dir / args.checkpoint_name
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "en_vocab_size": len(en_vocab),
            "ch_vocab_size": len(ch_vocab),
            "hidden_size": args.hidden_size,
            "max_seq_length": args.max_seq_length,
        },
        checkpoint_path,
    )
    log(f"checkpoint saved to: {checkpoint_path}")


def load_checkpoint(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict)
    log(f"checkpoint loaded: {checkpoint_path}")
    return checkpoint

In [ ]:
def ids_to_english(ids, vocab):
    tokens = []
    for idx in ids:
        if idx == PAD_ID or idx == EOS_ID:
            break
        if idx == SOS_ID:
            continue
        tokens.append(vocab[idx])
    return " ".join(tokens)


def ids_to_chinese(ids, vocab):
    chars = []
    for idx in ids:
        if idx == PAD_ID or idx == EOS_ID:
            break
        if idx == SOS_ID:
            continue
        chars.append(vocab[idx])
    return "".join(chars)


@torch.no_grad()
def compute_eval_loss(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    for batch in data_loader:
        src = batch["encoder_data"].to(device)
        tgt = batch["decoder_data"].to(device)
        label = batch.get("target_data")
        if label is None:
            continue
        label = label.to(device)
        logits = model(src, tgt)
        loss = criterion(logits.reshape(-1, logits.size(-1)), label.reshape(-1))
        total_loss += loss.item()
        total_batches += 1
    if total_batches == 0:
        return 0.0
    return total_loss / total_batches


@torch.no_grad()
def compute_eval_metrics(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    token_correct = 0
    token_total = 0
    seq_correct = 0
    seq_total = 0

    for batch in data_loader:
        src = batch["encoder_data"].to(device)
        tgt = batch["decoder_data"].to(device)
        label = batch["target_data"].to(device)

        logits = model(src, tgt)
        loss = criterion(logits.reshape(-1, logits.size(-1)), label.reshape(-1))
        total_loss += loss.item()
        total_batches += 1

        pred = model.greedy_decode(src, label.size(1))
        aligned_pred = torch.full_like(label, PAD_ID)
        copy_len = min(pred.size(1), label.size(1))
        aligned_pred[:, :copy_len] = pred[:, :copy_len]

        valid_mask = label.ne(PAD_ID)
        token_correct += ((aligned_pred == label) & valid_mask).sum().item()
        token_total += valid_mask.sum().item()

        seq_match = ((aligned_pred == label) | ~valid_mask).all(dim=1)
        seq_correct += seq_match.sum().item()
        seq_total += label.size(0)

    mean_loss = total_loss / total_batches if total_batches else 0.0
    token_acc = token_correct / token_total if token_total else 0.0
    seq_acc = seq_correct / seq_total if seq_total else 0.0
    return mean_loss, token_acc, seq_acc


def train_model(args):
    set_seed(args.seed)
    ensure_preprocessed(args)
    device = resolve_device(args.device)
    log(f"using device: {device}")

    train_loader = create_dataloader(args, is_training=True)
    eval_loader = create_dataloader(args, is_training=False)
    model, en_vocab, ch_vocab = build_model(args)
    model = model.to(device)
    log(f"train steps per epoch: {len(train_loader)}")
    log(f"hidden_size: {args.hidden_size}, batch_size: {args.batch_size}, epochs: {args.num_epochs}")

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate, betas=(0.9, 0.98))

    for epoch in range(1, args.num_epochs + 1):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader, start=1):
            src = batch["encoder_data"].to(device)
            tgt = batch["decoder_data"].to(device)
            label = batch["target_data"].to(device)

            optimizer.zero_grad()
            logits = model(src, tgt)
            loss = criterion(logits.reshape(-1, logits.size(-1)), label.reshape(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_loss = total_loss / max(len(train_loader), 1)
        eval_loss, token_acc, seq_acc = compute_eval_metrics(model, eval_loader, criterion, device)
        log(f"epoch {epoch}/{args.num_epochs} train_loss {train_loss:.6f} eval_loss {eval_loss:.6f} token_acc {token_acc:.4f} seq_acc {seq_acc:.4f}")

    save_checkpoint(model, args, en_vocab, ch_vocab)


@torch.no_grad()
def evaluate_model(args):
    ensure_preprocessed(args)
    device = resolve_device(args.device)
    log(f"using device: {device}")

    eval_loader = create_dataloader(args, is_training=False)
    model, en_vocab, ch_vocab = build_model(args)
    model = model.to(device)
    checkpoint_path = args.checkpoint_dir / args.checkpoint_name
    load_checkpoint(model, checkpoint_path, device)
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    eval_loss, token_acc, seq_acc = compute_eval_metrics(model, eval_loader, criterion, device)
    log(f"eval loss: {eval_loss:.6f}")
    log(f"token accuracy: {token_acc:.4f}")
    log(f"sequence accuracy: {seq_acc:.4f}")
    log(f"eval samples: {len(eval_loader.dataset)}")

    shown = 0
    for idx, batch in enumerate(eval_loader, start=1):
        src = batch["encoder_data"].to(device)
        pred = model.greedy_decode(src, args.max_seq_length).cpu()

        english = ids_to_english(batch["encoder_data"][0].tolist(), en_vocab)
        expect_chinese = ids_to_chinese(batch["decoder_data"][0].tolist(), ch_vocab)
        predict_chinese = ids_to_chinese(pred[0].tolist(), ch_vocab)

        log(f"eval sample {idx}")
        log(f"English: {english}")
        log(f"expect Chinese: {expect_chinese}")
        log(f"predict Chinese: {predict_chinese}")
        shown += 1
        if shown >= args.eval_log_samples:
            break


@torch.no_grad()
def predict_one_sentence(args):
    ensure_preprocessed(args)
    device = resolve_device(args.device)
    log(f"using device: {device}")

    model, en_vocab, ch_vocab = build_model(args)
    model = model.to(device)
    checkpoint_path = args.checkpoint_dir / args.checkpoint_name
    load_checkpoint(model, checkpoint_path, device)
    model.eval()

    sentence = normalize_english(args.predict_sentence)
    en2id = {token: idx for idx, token in enumerate(en_vocab)}
    tokens = sentence.split()
    ids = [SOS_ID] + [en2id.get(token, PAD_ID) for token in tokens] + [PAD_ID]
    ids = pad_or_truncate(ids, args.max_seq_length)
    src = torch.tensor([ids[1:]], dtype=torch.long, device=device)
    pred = model.greedy_decode(src, args.max_seq_length).cpu()[0].tolist()

    log(f"input English: {sentence}")
    log(f"predict Chinese: {ids_to_chinese(pred, ch_vocab)}")


def run_pipeline(args):
    log_path = setup_log_file(args)
    log(f"log file: {log_path}")
    log("start experiment")
    log("config: " + json.dumps(config_to_dict(args), ensure_ascii=False))
    
    if args.command == "all":
        log("run mode: preprocess -> train -> eval -> predict")
        preprocess_dataset(args)
        train_model(args)
        evaluate_model(args)
        predict_one_sentence(args)
    elif args.command == "preprocess":
        preprocess_dataset(args)
    elif args.command == "train":
        train_model(args)
    elif args.command == "eval":
        evaluate_model(args)
    elif args.command == "predict":
        predict_one_sentence(args)
    else:
        raise ValueError(f"unknown command: {args.command}")
    
    log("experiment finished")

## 运行

直接修改 `RUN_CONFIG`，默认 `command="all"` 会顺序跑完 `preprocess -> train -> eval -> predict`。每次运行都会在 `./logs/` 下生成一个新的日志文件。

In [ ]:
args = build_args(RUN_CONFIG)
run_pipeline(args)